Recreacion del codigo para ejecutar el algoritmo txmeans, puediendolo depurar de una manera mas eficiente

In [20]:
import sys
sys.path.insert(0, r'/home/adrian/Escritorio/TFG/TXMeans/code/')

In [2]:
import os

print(os.getcwd())

/home/adrian/Escritorio/TFG/TXMeans/code/test_algoritmos


In [33]:
from algorithms.txmeans import *
from generators.datamanager import *
from validation.validation_measures import *
# from generators.datagenerator import *


Leer y modificar el dataset

In [75]:
def read_uci_data(filename, class_index=0, delimiter=',', missing_symbol='?', header=True, skipcolumnsindex=set()):
    

    df = pd.read_csv(filename, skipinitialspace=True)
    df[['ID', 'CLASS', 'EVENTS']] = df['ID;CLASS;EVENTS'].str.split(';', expand=True)
    df['EVENTS'] = df['EVENTS'].str.replace(' ', ';')
    index_mode = dict()
    print(df.columns)
    for k, index in zip(df.columns, range(0, len(df.columns))):
        df[k] = df[k].replace('?', np.nan)
        mode_value = mode(df[k])[0][0]
        df[k] = df[k].fillna(mode_value)
        index_mode[index] = mode_value

    baskets = list()

    data = open(filename, 'r')

    map_item_newitem = dict()
    map_newitem_item = dict()
    map_class_newclass = dict()
    map_newclass_class = dict()

    if header:
        data.readline()
    for row in data:
        categories = row.rstrip().split(delimiter)
        basket = list()
        basket_class = None
        for index in range(0, len(categories)):

            if index in skipcolumnsindex:
                continue

            if index == class_index:
                cclass = categories[index]
                if categories[index] not in map_class_newclass:
                    newclass = len(map_class_newclass)
                    map_class_newclass[cclass] = newclass
                    map_newclass_class[newclass] = cclass
                basket_class = map_class_newclass[cclass]
                continue

            if categories[index] == missing_symbol:
                categories[index] = index_mode[index]

            item = (index, categories[index])
            if item not in map_item_newitem:
                newitem = len(map_item_newitem)
                map_item_newitem[item] = newitem
                map_newitem_item[newitem] = item
            newitem = map_item_newitem[item]
            basket.append(newitem)

        if len(basket) > 0:
            baskets.append((basket, basket_class))

    data.close()

    maps = {
        'map_item_newitem': map_item_newitem,
        'map_newitem_item': map_newitem_item,
        'map_class_newclass': map_class_newclass,
        'map_newclass_class': map_newitem_item,
    }
    
    return baskets, maps

In [76]:
path = '../../../dataset_pp/'
dataset_name = 'P0.O0_tx.data'
filename = path + dataset_name
txmeans = TXmeans()
	
filename = path + dataset_name
class_index = 1
skipcolumnsindex = set({0})
	
baskets_real_labels, maps = read_uci_data(filename, delimiter=";",class_index=class_index, skipcolumnsindex=skipcolumnsindex)

print( dataset_name, len(baskets_real_labels))

Index(['ID;CLASS;EVENTS', 'ID', 'CLASS', 'EVENTS'], dtype='object')


/tmp/ipykernel_84619/1142183890.py:11: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  mode_value = mode(df[k])[0][0]
/tmp/ipykernel_84619/1142183890.py:11: DeprecationWarning: Support for non-numeric arrays has been deprecated as of SciPy 1.9.0 and will be removed in 1.11.0. `pandas.DataFrame.mode` can be used instead, see https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mode.html.
  mode_value = mode(df[k])[0][0]


P0.O0_tx.data 97485


Preparar el dataset para ser ejecutado, añadiendo las transacciones en baskets, modificandolas a bits

In [71]:
baskets_list = list()
real_labels = list()
count = 0
for basket, label in baskets_real_labels:
    baskets_list.append(basket)
    real_labels.append(label)
    count += 1
baskets_list, map_newitem_item, map_item_newitem = remap_items(baskets_list)
baskets_list = basket_list_to_bitarray(baskets_list, len(map_newitem_item))

nbaskets = len(baskets_list)
nitems = count_items(baskets_list)

In [72]:

print("Number of baskets: ", nbaskets)
print("Number of items: ", nitems)

Number of baskets:  8124
Number of items:  116


Ejecucion del algoritmo txmeans

In [5]:
start_time = datetime.datetime.now()

nsample = sample_size(nbaskets, 0.05, conf_level=0.99, prob=0.5)
txmeans.fit(baskets_list, nbaskets, nitems, random_sample=nsample)


end_time = datetime.datetime.now()
running_time = end_time - start_time

Resultados Pd cada vez que se ejecuta el algoritmo los resultasdos varian 

In [6]:
res = txmeans.clustering
#iter_count = bicartd.iter_count
pred_labels = [0] * len(real_labels)
baskets_clusters = list()
for cluster, label in zip(res, range(0, len(res))):
    cluster_list = basket_bitarray_to_list(cluster['cluster']).values()
    for bid in cluster['cluster']:
        pred_labels[bid] = label
        baskets_clusters.append(cluster_list)

print('delta_k', delta_k(real_labels, pred_labels))
print('normalized_mutual_info_score', normalized_mutual_info_score(real_labels, pred_labels))
print('purity', purity(real_labels, pred_labels))
print('running_time', running_time)

delta_k 3
normalized_mutual_info_score 0.35213904476629476
purity 0.8906942392909897
running_time 0:00:00.099383
